In [1]:
!pip install transformers torch accelerate scikit-learn peft trl datasets -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import os
import json
import torch
import gc
from collections import Counter
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig
from sklearn.metrics import f1_score

# 데이터 로드
with open('../context_conditions.json', encoding='utf-8') as f:
    full_dataset = json.load(f)

train_data  = [d for d in full_dataset if d['split'] == 'train' and d['useful'] is not None]
dataset     = [d for d in full_dataset if d['split'] == 'dev']

print(f"train 샘플 수: {len(train_data)}")
print(f"dev 샘플 수  : {len(dataset)}")
print(f"train 레이블 분포: {Counter(d['label'] for d in train_data)}")

train 샘플 수: 4890
dev 샘플 수  : 1447
train 레이블 분포: Counter({'comment': 3508, 'support': 642, 'query': 373, 'deny': 367})


In [5]:
CONDITIONS   = ["reply_only", "useful", "irrelevant", "conflicting", "mixed", "lexical"]
LABEL_MAP    = {"support": 0, "deny": 1, "query": 2, "comment": 3}
VALID_LABELS = {"support", "deny", "query", "comment"}

SYSTEM_PROMPT = """You are a stance classification expert for social media discussions about rumours.

Classify the stance of the TARGET reply using exactly one of these four labels:

- support: The reply explicitly states the rumour IS true or confirmed.
- deny: The reply explicitly states the rumour IS false or fabricated.
- query: The reply asks for sources, evidence, or verification.
- comment: Everything else. The reply does not directly address whether the rumour is true or false.

Respond with ONLY one word: support, deny, query, or comment. No explanation."""


def build_prompt(text: str) -> list:
    cleaned = text
    cleaned = cleaned.replace("[Source]",     "Rumour post:")
    cleaned = cleaned.replace("[Context]",    "Previous reply:")
    cleaned = cleaned.replace("[Misleading]", "Another reply:")
    cleaned = cleaned.replace("[Target]",     "Reply to classify:")
    user_content = (
        f"Read the following and classify the stance of the 'Reply to classify'.\n\n"
        f"{cleaned}\n\n"
        f"Stance label (support / deny / query / comment):"
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]


def format_sample(d, condition='useful'):
    text = d[condition]
    if condition == 'mixed':
        text = text.replace("[Source]",     "Rumour post:")
        text = text.replace("[Context]",    "Previous reply:")
        text = text.replace("[Misleading]", "Another reply:")
        text = text.replace("[Target]",     "Reply to classify:")
    else:
        text = text.replace("[Source]",  "Rumour post:")
        text = text.replace("[Context]", "Previous reply:")
        text = text.replace("[Target]",  "Reply to classify:")
    user_content = (
        f"Read the following and classify the stance of the 'Reply to classify'.\n\n"
        f"{text}\n\n"
        f"Stance label (support / deny / query / comment):"
    )
    return {
        "text": (
            f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{user_content}<|im_end|>\n"
            f"<|im_start|>assistant\n{d['label']}<|im_end|>"
        )
    }


def evaluate(result: dict) -> dict:
    golds = result["golds"]
    preds = result["preds"]
    if len(golds) == 0:
        return {"macro_f1": 0.0, "per_class": [0.0] * 4}
    macro_f1  = f1_score(golds, preds, average="macro",  zero_division=0)
    per_class = f1_score(golds, preds, average=None, labels=[0,1,2,3], zero_division=0)
    return {"macro_f1": macro_f1, "per_class": per_class.tolist()}


def run_condition(condition: str, data: list, predict_fn) -> dict:
    golds, preds  = [], []
    invalid_count = 0
    skipped_count = 0
    total         = len(data)
    for i, d in enumerate(data):
        if d[condition] is None:
            skipped_count += 1
            continue
        pred = predict_fn(d[condition])
        if pred == "invalid":
            invalid_count += 1
            continue
        golds.append(LABEL_MAP[d["label"]])
        preds.append(LABEL_MAP[pred])
        if (i + 1) % 100 == 0:
            print(f"  [{condition}] {i+1}/{total} | 유효 {len(golds)} | invalid {invalid_count}")
    return {
        "golds":         golds,
        "preds":         preds,
        "invalid_count": invalid_count,
        "skipped_count": skipped_count,
        "n":             len(golds),
    }


def make_predict_fn(model, tokenizer):
    def predict(text: str, max_new_tokens: int = 10) -> str:
        messages   = build_prompt(text)
        text_input = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )
        input_ids = tokenizer(text_input, return_tensors="pt").input_ids.to(model.device)
        with torch.no_grad():
            output_ids = model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=1.0,
                pad_token_id=tokenizer.eos_token_id,
            )
        new_tokens = output_ids[0][input_ids.shape[-1]:]
        raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()
        for label in VALID_LABELS:
            if label in raw_output:
                return label
        return "invalid"
    return predict

print("함수 정의 완료!")

함수 정의 완료!


In [6]:
adv_train_data = []

for d in train_data:
    # useful 항상 포함
    adv_train_data.append(format_sample(d, 'useful'))
    # conflicting 있으면 추가
    if d.get('conflicting') is not None:
        adv_train_data.append(format_sample(d, 'conflicting'))
    # mixed 있으면 추가
    if d.get('mixed') is not None:
        adv_train_data.append(format_sample(d, 'mixed'))

adv_dataset = Dataset.from_list(adv_train_data)

print(f"기존 train 데이터  : {len(train_data)}개")
print(f"adversarial 데이터: {len(adv_dataset)}개")
print(f"레이블 분포:")
labels = [d['label'] for d in train_data for _ in range(
    1 + (1 if d.get('conflicting') else 0) + (1 if d.get('mixed') else 0)
)]
print(Counter(labels))

기존 train 데이터  : 4890개
adversarial 데이터: 12878개
레이블 분포:
Counter({'comment': 9490, 'support': 1422, 'query': 1029, 'deny': 937})


In [7]:
# 1.5B Adversarial 파인튜닝
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer_adv = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer_adv.pad_token = tokenizer_adv.eos_token

model_adv = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model_adv = get_peft_model(model_adv, lora_config)
model_adv.print_trainable_parameters()

sft_config = SFTConfig(
    output_dir=os.path.expanduser("~/qwen_1.5b_adv"),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    warmup_steps=100,
    lr_scheduler_type="cosine",
    report_to="none",
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model_adv,
    args=sft_config,
    train_dataset=adv_dataset,
    processing_class=tokenizer_adv,
)

print("1.5B Adversarial 파인튜닝 시작...")
trainer.train()
print("파인튜닝 완료!")

save_path = os.path.expanduser("~/qwen_1.5b_adv/final")
model_adv.save_pretrained(save_path)
tokenizer_adv.save_pretrained(save_path)
print(f"저장 완료: {save_path}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 521.94it/s]


trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


Tokenizing train dataset: 100%|██████████| 12878/12878 [00:06<00:00, 1883.21 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


1.5B Adversarial 파인튜닝 시작...


Step,Training Loss
50,3.310993
100,1.916861
150,1.331307
200,1.274364
250,1.229293
300,1.208710
350,1.185332
400,1.195419
450,1.115930
500,1.074426


파인튜닝 완료!
저장 완료: /home/ubuntu/qwen_1.5b_adv/final


In [ ]:
# adversarial 파인튜닝된 1.5B 로드 + dev 평가
torch.cuda.empty_cache()
gc.collect()

ft_path_adv = os.path.expanduser("~/qwen_1.5b_adv/final")
BASE_MODEL  = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer_adv_eval = AutoTokenizer.from_pretrained(ft_path_adv)
base_model_adv     = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)
model_adv_eval = PeftModel.from_pretrained(base_model_adv, ft_path_adv)
model_adv_eval.eval()
print("Adversarial 1.5B 로드 완료!")

predict_adv = make_predict_fn(model_adv_eval, tokenizer_adv_eval)

# dev 평가
all_results_adv = {}
for condition in CONDITIONS:
    print(f"\n{'='*55}")
    print(f"  조건: {condition}")
    print(f"{'='*55}")
    result  = run_condition(condition, dataset, predict_fn=predict_adv)
    metrics = evaluate(result)
    all_results_adv[condition] = {"raw": result, "metrics": metrics}
    pc = metrics["per_class"]
    print(f"  완료 → n={result['n']} | invalid={result['invalid_count']}")
    print(f"  macro-F1 : {metrics['macro_f1']:.4f}")
    print(f"  support={pc[0]:.3f}  deny={pc[1]:.3f}  query={pc[2]:.3f}  comment={pc[3]:.3f}")

# 결과 저장
save_data_adv = {}
for condition, data in all_results_adv.items():
    save_data_adv[condition] = {
        "golds":        data["raw"]["golds"],
        "preds":        data["raw"]["preds"],
        "invalid_count":data["raw"]["invalid_count"],
        "n":            data["raw"]["n"],
        "macro_f1":     data["metrics"]["macro_f1"],
        "per_class_f1": data["metrics"]["per_class"],
    }
with open('experiment_results_1.5B_adv.json', 'w', encoding='utf-8') as f:
    json.dump(save_data_adv, f, ensure_ascii=False, indent=2)
print("\n저장 완료: experiment_results_1.5B_adv.json")

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 510.35it/s]
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Adversarial 1.5B 로드 완료!

  조건: reply_only
  [reply_only] 100/1447 | 유효 100 | invalid 0
  [reply_only] 200/1447 | 유효 195 | invalid 5
  [reply_only] 300/1447 | 유효 293 | invalid 7
  [reply_only] 400/1447 | 유효 393 | invalid 7
  [reply_only] 500/1447 | 유효 493 | invalid 7
  [reply_only] 600/1447 | 유효 593 | invalid 7
  [reply_only] 700/1447 | 유효 693 | invalid 7
  [reply_only] 800/1447 | 유효 793 | invalid 7
  [reply_only] 900/1447 | 유효 893 | invalid 7
  [reply_only] 1000/1447 | 유효 993 | invalid 7
  [reply_only] 1100/1447 | 유효 1093 | invalid 7
  [reply_only] 1200/1447 | 유효 1193 | invalid 7
  [reply_only] 1300/1447 | 유효 1293 | invalid 7
